# Decision Tree From Scratch (Simple Implementation)
This notebook implements a basic binary decision tree using Gini impurity.

## Step 1: Create Dataset

In [1]:
import numpy as np

# Outlook: Sunny=0, Overcast=1, Rain=2
X = np.array([
    [0,30],[0,32],[1,28],[2,22],
    [2,20],[2,18],[1,24],[0,21]
])
# Play: No=0, Yes=1
y = np.array([0,0,1,1,1,0,1,1])

print(X)
print(y)


[[ 0 30]
 [ 0 32]
 [ 1 28]
 [ 2 22]
 [ 2 20]
 [ 2 18]
 [ 1 24]
 [ 0 21]]
[0 0 1 1 1 0 1 1]


## Step 2: Gini Impurity
Formula: $Gini = 1-\sum p_i^2$

In [2]:
def gini(labels):
    classes, counts = np.unique(labels, return_counts=True)
    impurity = 1
    total = len(labels)
    for count in counts:
        p = count/total
        impurity -= p**2
    return impurity

print(gini(np.array([1,1,1,1])))
print(gini(np.array([1,1,0,0])))


0.0
0.5


## Step 3: Split Dataset

In [3]:
def split_dataset(X,y,feature,threshold):
    left = X[:,feature] <= threshold
    right = ~left
    return X[left], X[right], y[left], y[right]

XL,XR,yL,yR=split_dataset(X,y,1,25)
print(yL,yR)


[1 1 0 1 1] [0 0 1]


## Step 4: Weighted Gini

In [4]:
def weighted_gini(y_left,y_right):
    total=len(y_left)+len(y_right)
    return len(y_left)/total*gini(y_left)+len(y_right)/total*gini(y_right)

print(weighted_gini(yL,yR))


0.3666666666666666


## Step 5: Find Best Split

In [5]:
def best_split(X,y):
    best_feature,best_threshold=None,None
    best_score=float('inf')
    for feature in range(X.shape[1]):
        for threshold in np.unique(X[:,feature]):
            XL,XR,yL,yR=split_dataset(X,y,feature,threshold)
            if len(yL)==0 or len(yR)==0:
                continue
            score=weighted_gini(yL,yR)
            if score<best_score:
                best_score=score
                best_feature=feature
                best_threshold=threshold
    return best_feature,best_threshold

print(best_split(X,y))


(1, np.int64(28))


## Step 6: Node Class

In [6]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value


## Step 7: Build Tree

In [7]:
def build_tree(X,y,depth=0,max_depth=3):
    if len(set(y))==1:
        return Node(value=y[0])
    if depth>=max_depth:
        values,counts=np.unique(y,return_counts=True)
        return Node(value=values[np.argmax(counts)])
    feature,threshold=best_split(X,y)
    XL,XR,yL,yR=split_dataset(X,y,feature,threshold)
    left=build_tree(XL,yL,depth+1,max_depth)
    right=build_tree(XR,yR,depth+1,max_depth)
    return Node(feature,threshold,left,right)

tree=build_tree(X,y)
print("Tree built")


Tree built


## Step 8: Prediction

In [8]:
def predict(node,sample):
    if node.value is not None:
        return node.value
    if sample[node.feature] <= node.threshold:
        return predict(node.left,sample)
    return predict(node.right,sample)

sample=np.array([2,21])
print("Prediction:",predict(tree,sample))


Prediction: 1


## Summary

**Training**
1. Try every feature.
2. Try every threshold.
3. Compute weighted Gini.
4. Choose the split with the lowest impurity.
5. Repeat recursively.

**Prediction**
Start at the root and follow the comparisons until reaching a leaf node.
